In lecture 10 of mcs 320, we time Python functions and make them more efficient by vectorization and by Cython.

# 1. Timing Python functions

This lecture follows Chapter 3 of Sage for Power Users by William Stein.
We consider the computation of a floating-point approximation
of a sum of square roots.  Such sums occur in numerical integration.
One way to approximate $\pi/4$ is as below.

In [1]:
def python_sum_symbolic(n):
    return float( sum(sqrt(1-(k/n)^2) for k in range(1, n+1)) )/n
4*python_sum_symbolic(1000)

3.1395554669110264

To benchmark the function, we can use timeit.
The command ``timeit`` in Python is good to measure
the execution time of small code snippets.

In [2]:
timeit('python_sum_symbolic(1000)')

5 loops, best of 3: 58.8 ms per loop

For longer execution times, we better use ``cputime()``.

In [3]:
t1 = cputime() # cpu time since Sage started
python_sum_symbolic(10^4)
ct1 = cputime(t1) # cpu time since t1
print('time for python_sum_symbolic :', ct1)

time for python_sum_symbolic : 1.859


# 2. Use numerical functions

The first reason that the function is so slow is because we use
the symbolic sqrt function. 
We will use the sqrt function of the math module in Python.
The explicit conversion to float is no longer needed because
the this sqrt returns a floating-point number.

In [4]:
def python_sum(n):
    from math import sqrt
    return sum( sqrt(1-(k/n)^2) for k in range(1, n+1) )/n
4*python_sum(1000)

3.139555466911023

In [5]:
t2 = cputime()
python_sum(10^4)
ct2 = cputime(t2)
print('time for python_sum :', ct2)

time for python_sum : 0.016000000000000014


In [6]:
timeit('python_sum(10^4)')

25 loops, best of 3: 21.3 ms per loop

In [7]:
print('speedup :', ct1/ct2)

speedup : 116.1874999999999


# 3. Vectorize

Instead of the sum we could apply vectorization with numpy.

In [8]:
def numpy_sum(n): 
    from numpy import sqrt, sum, arange
    x = arange(n)/float(n)
    return sum(sqrt(1-x**2))/n
4*numpy_sum(1000)

3.1435554669110277

In [9]:
timeit('numpy_sum(10^4)')

625 loops, best of 3: 101 μs per loop

In [10]:
# To compare against pure Python, we sum a million times.
t3 = cputime()
python_sum(10^6)
ct3 = cputime(t3)
print('time for python_sum :', ct3)
t4 = cputime()
numpy_sum(10^6)
ct4 = cputime(t4)
print('time for numpy_sum :', ct4)

time for python_sum : 2.1249999999999982
time for numpy_sum : 0.031000000000000583


In [11]:
print('speedup :', ct3/ct4)

speedup : 68.54838709677284


# 4. Cythonize

In [12]:
%%cython
def cython_sum(n):
    from math import sqrt
    return sum( sqrt(1-(k/n)**2) for k in range(1, n+1) )/n

The advantage of Cython is that the code is almost identical to Python.
The two links returned when evaluating a cell with Cython code are the generated C code
and an html file with the annotated version of the Cython program.
In this particular example we had to replace the ^ of the python_sum with **.

In [13]:
print(4*cython_sum(1000))

3.139555466911023


In [14]:
timeit('cython_sum(10^4)')

5 loops, best of 3: 32.8 ms per loop

In [15]:
t5 = cputime()
cython_sum(10^6)
ct5 = cputime(t5)
print('time for the cython_sum :', ct5)

time for the cython_sum : 3.2029999999999976


In [16]:
%%cython
cdef extern from "math.h":
    double sqrt(double)

def cython_sum_typed(long n):
    cdef long k
    return sum( sqrt(1-(k/float(n))**2) for k in range(1, n+1) )/n

In [17]:
print(4*cython_sum_typed(1000))

3.139555466911023


In [18]:
timeit('cython_sum_typed(10^4)')

625 loops, best of 3: 1.11 ms per loop

In [19]:
t6 = cputime()
cython_sum_typed(10^6)
ct6 = cputime(t6)
print('time for cython_sum_typed :', ct6)

time for cython_sum_typed : 0.10999999999999943


In [20]:
print('speedup :', ct5/ct6)

speedup : 29.11818181818195
